In [ ]:
#将下载的多时间段nc文件进行拼接，不是所有的都需要拼接
import os
import xarray as xr

# 定义文件夹路径
folder1 = r'xx\xxx'  # 包含原始.nc文件的文件夹
folder2 = r'xx\xxxxx'  # 保存拼接后的.nc文件的文件夹

# 创建folder2（若没有）
if not os.path.exists(folder2):
    os.makedirs(folder2)

# 获取folder1下所有的.nc文件
nc_files = [os.path.join(folder1, f) for f in os.listdir(folder1) if f.endswith('.nc')]

# 使用xarray打开所有.nc文件并按时间维度拼接
try:
    ds = xr.open_mfdataset(nc_files, combine='by_coords')
except Exception as e:
    raise ValueError(f"文件拼接失败，请检查文件内容是否一致或时间维度是否正确")

# 定义输出文件路径
output_path = os.path.join(folder2, 'xx\xxx.nc')   # ！！！！！修改输出文件名称和路径

# 保存拼接后的数据集到新的.nc文件
ds.to_netcdf(output_path)

print(f"拼接后的文件已保存到: {output_path}")

In [ ]:
#计算日pet，先准备好日tas、tasmax、tasmin、hurs、sfcWind、rsds数据
import xarray as xr
import numpy as np
import os

# 设置文件夹路径，包含上面5个变量，且每个nc文件都用相对应的变量名称命名（和下面的“# 分块加载数据”保持一致）
folder_path = r"xx\xx"

# 常量定义
STEFAN_BOLTZMANN_CONSTANT = 4.903e-9  # MJ/m²/°C⁴/day
ALBEDO = 0.23  # 地表反照率
PSYCHROMETRIC_CONSTANT = 0.0665  # 心理常数 (kPa/°C)

# 辅助函数
def calc_vapor_pressure(temp):
    """计算饱和水汽压 (kPa)"""
    return 0.6108 * np.exp((17.27 * (temp - 273.15)) / ((temp - 273.15) + 237.3))

def calc_slope_of_vapor_pressure_curve(temp):
    """计算饱和水汽压随温度的变化率 (kPa/°C)"""
    return (4098 * calc_vapor_pressure(temp)) / ((temp - 273.15) + 237.3)**2

# 分块加载数据
tas = xr.open_dataset(os.path.join(folder_path, "tas.nc"), chunks={"time": 100})["tas"]
tasmax = xr.open_dataset(os.path.join(folder_path, "tasmax.nc"), chunks={"time": 100})["tasmax"]
tasmin = xr.open_dataset(os.path.join(folder_path, "tasmin.nc"), chunks={"time": 100})["tasmin"]
hurs = xr.open_dataset(os.path.join(folder_path, "hurs.nc"), chunks={"time": 100})["hurs"]
sfcWind = xr.open_dataset(os.path.join(folder_path, "sfcWind.nc"), chunks={"time": 100})["sfcWind"]
rsds = xr.open_dataset(os.path.join(folder_path, "rsds.nc"), chunks={"time": 100})["rsds"]

# 计算各变量
T_mean = tas - 273.15  # 平均气温 (°C)
T_max = tasmax - 273.15  # 最高气温 (°C)
T_min = tasmin - 273.15  # 最低气温 (°C)

# 计算饱和水汽压和实际水汽压
e_s_max = calc_vapor_pressure(tasmax)
e_s_min = calc_vapor_pressure(tasmin)
e_s = (e_s_max + e_s_min) / 2  # 平均饱和水汽压 (kPa)
e_a = calc_vapor_pressure(tas) * (hurs / 100)  # 实际水汽压 (kPa)

# 计算短波辐射 (MJ/m²/day)
R_s = rsds * 0.0864  # 将短波辐射从 W/m² 转换为 MJ/m²/day

# 计算长波辐射 (MJ/m²/day)
R_nl = (
    STEFAN_BOLTZMANN_CONSTANT *
    ((T_max + 273.15)**4 + (T_min + 273.15)**4) *
    (0.34 - 0.14 * np.sqrt(e_a)) *
    (1.35 * (R_s / 24) - 0.35)
)

# 计算净辐射 (MJ/m²/day)
R_n = (1 - ALBEDO) * R_s - R_nl

# 计算斜率
delta = calc_slope_of_vapor_pressure_curve(tas)

# 计算PET (mm/day)
PET = (
    (delta * (R_n - 0) + PSYCHROMETRIC_CONSTANT * (900 / (T_mean + 273.15)) * sfcWind * (e_s - e_a)) /
    (delta + PSYCHROMETRIC_CONSTANT * (1 + 0.34 * sfcWind))
) 

# 保存结果
PET.name = "pet"
PET.attrs["units"] = "mm"
PET.attrs["long_name"] = "Potential Evapotranspiration"

# 使用 compute() 强制计算并释放内存
PET = PET.compute()

# 保存到文件
PET.to_netcdf(os.path.join(folder_path, "pet.nc"))

print("PET计算完成并保存到文件夹中")

In [ ]:
#日PET累加计算年PET
import os
import xarray as xr

# 定义输入文件路径和输出文件目录
input_file = r"xxx\\pet.nc"  # 替换为实际文件路径
output_dir = r"xxx\\pet"  # 输出目录

# 创建输出目录output_dir（若不存在）
os.makedirs(output_dir, exist_ok=True)

# 打开NC文件
ds = xr.open_dataset(input_file)

# 确保文件中有'time'维度和'pet'变量
if "time" not in ds.dims or "pet" not in ds.variables:
    raise ValueError("输入文件中缺少'time'维度或'pet'变量")

# 将时间转换为年份和天数
ds['year'] = ds['time'].dt.year
ds['day'] = ds['time'].dt.dayofyear

# 计算每年的总潜在蒸散发
for year in range(2015, 2101):  # 2015年到2100年
#for year in range(1985, 2015):  # 1985年到2014年    
    # 筛选当前年份的数据
    yearly_data = ds['pet'].sel(time=ds['year'] == year)

    # 初始化年总潜在蒸散发
    annual_sum = xr.zeros_like(yearly_data.isel(time=0))

    # 逐日累加潜在蒸散发
    for day in range(1, 367):  # 1到366天（考虑闰年）
        # 筛选当前日的数据
        daily_data = yearly_data.sel(time=yearly_data['time'].dt.dayofyear == day)
        if len(daily_data) > 0:
            annual_sum += daily_data.sum(dim='time')

    # 创建一个新的数据集用于存储
    annual_ds = xr.Dataset(
        {"pet": annual_sum},
        attrs=ds.attrs
    )

    # 更新变量的units属性为"mm"
    annual_ds['pet'].attrs['units'] = 'mm'

    # 为新的数据集添加坐标信息
    for coord in ds.coords:
        if coord != 'time':  # 排除time维度
            annual_ds[coord] = ds[coord]

    # 保存到新的NC文件
    output_file = os.path.join(output_dir, f"pet_{year}.nc")
    annual_ds.to_netcdf(output_file)
    print(f"保存 {year} 的总和到文件 {output_file}")

print("处理完成！")

In [ ]:
#将文件夹下的所有nc文件的分辨率统一为8640, 4320
import os
import glob
import xarray as xr
import numpy as np

# 定义文件夹路径
input_folder = r'xxx\\pet'
output_folder = r'xxx\\pet'

# 创建输出目录
os.makedirs(output_folder, exist_ok=True)

# 目标网格
target_lon_count, target_lat_count = 8640, 4320

# 计算目标经纬度
new_lon = np.linspace(-180, 180, target_lon_count + 1)[:-1]
new_lat = np.linspace(-90, 90, target_lat_count + 1)[:-1]

# 获取所有 .nc 文件
nc_files = glob.glob(os.path.join(input_folder, '*.nc'))

# 处理所有文件
for nc_file in nc_files:
    ds = xr.open_dataset(nc_file)

    # 确保 lon 在 [-180, 180] 范围
    ds = ds.assign_coords(lon=((ds.lon + 180) % 360 - 180)).sortby('lon')

    # 处理 lon=0 的情况
    if 0 not in ds.lon.values:
        ds_zero = ds.interp(lon=0, method="linear")  # 线性插值
    else:
        ds_zero = ds.sel(lon=0).copy(deep=True)

    ds_zero = ds_zero.assign_coords(lon=360)
    ds_ext = xr.concat([ds, ds_zero], dim="lon")

    # 插值，这里的变量是pet
    new_data = ds_ext.pet.interp(lat=new_lat, lon=new_lon, method='linear')

    # 保存处理后的数据
    filename = os.path.basename(nc_file)
    output_path = os.path.join(output_folder, filename)
    new_data.to_netcdf(output_path)

    ds.close()

print("批量处理完成！")

In [ ]:
#计算7个模式的平均值
import os
import xarray as xr
import numpy as np
from tqdm import tqdm  # 用于显示进度条

# 定义输入文件夹路径和输出文件夹路径
input_folder = r"xxx\\pet"
output_folder = r"xxx\\pet"

# 创建输出文件夹（如果不存在）
os.makedirs(output_folder, exist_ok=True)

# 获取所有子文件夹
sub_folders = [os.path.join(input_folder, sub) for sub in os.listdir(input_folder) if os.path.isdir(os.path.join(input_folder, sub))]

# 获取第一个子文件夹中的文件列表（所有子文件夹中的变量文件名称相同）
file_names = [f for f in os.listdir(sub_folders[0]) if f.endswith('.nc')]

# 遍历每个文件
for file_name in tqdm(file_names, desc="Processing files"):
    data_list = []  # 存储不同模式的同名文件数据

    # 遍历每个子文件夹，读取相同名称的文件
    for sub_folder in sub_folders:
        file_path = os.path.join(sub_folder, file_name)
        if os.path.exists(file_path):  # 确保文件存在
            ds = xr.open_dataset(file_path)
            data_list.append(ds)

    if data_list:
        reference_ds = data_list[0]

        # **计算多模式平均值**
        combined_data = xr.concat(data_list, dim="height")
        mean_data = combined_data.mean(dim="height")

        # **补充缺失的属性信息**
        mean_data.attrs = reference_ds.attrs
        for var in mean_data.data_vars:
            if var in reference_ds:
                mean_data[var].attrs = reference_ds[var].attrs

        # 保存到输出文件夹
        output_path = os.path.join(output_folder, file_name)
        mean_data.to_netcdf(output_path)
        print(f"保存 {file_name} 的平均值到 {output_path}")

print("处理完成！")